# GraphInstruct Quickstart (5 minutes)

This notebook walks you through:
1. Loading the 800-instruction benchmark
2. Inspecting the 6 complexity levels (L0–L5)
3. Running the D1, D4, D5 evaluation pipeline on a small mock model
4. Reading the resulting quality scores

**No API key needed.** This notebook uses a deterministic mock generator that
always emits the first reference solution — useful for verifying the eval
pipeline end-to-end without spending tokens.


## Setup

If you haven't installed yet:
```bash
pip install -e ..   # from the repo root
```

Windows users: set these env vars **before** launching Jupyter:
```bash
export KMP_DUPLICATE_LIB_OK=TRUE
export PYTHONIOENCODING=utf-8
```


In [ ]:
import os
import sys
from pathlib import Path

# Allow running this notebook either from examples/ or from repo root
REPO = Path.cwd()
if (REPO.name == 'examples'):
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('PYTHONIOENCODING', 'utf-8')
print(f'Repo root: {REPO}')


## 1. Load the 800-instruction benchmark

Each instruction is a (natural-language query, constraint specification, reference solutions) tuple.


In [ ]:
from graphinstruct.data_loader import load_all_levels

instructions = load_all_levels(data_dir=REPO / 'data' / 'instructions')
for L in sorted(instructions):
    print(f'  L{L}: {len(instructions[L])} instructions')
total = sum(len(v) for v in instructions.values())
print(f'\nTotal: {total} instructions across 6 levels')


## 2. Inspect a sample instruction

Let's look at one L1 instruction (single explicit constraint).


In [ ]:
sample = instructions[1][0]
print(f'ID: {sample.id}')
print(f'Level: L{sample.level}')
print(f'Instruction: {sample.instruction}')
print(f'Explicit constraints: {sample.explicit_constraints}')
print(f'Implicit constraints: {sample.implicit_constraints}')
print(f'Number of reference solutions: {len(sample.reference_solutions)}')


## 3. Run a deterministic mock generator

We'll use a mock LLM that just echoes the first reference solution. This
lets us verify the evaluation pipeline gives sensible numbers (it should
score near-perfect on D1 and D4 since outputs match the references).


In [ ]:
# Pick 5 instructions from each of L0/L1/L2/L3 (skip L4/L5 for speed)
mini = []
for L in (0, 1, 2, 3):
    for inst in instructions[L][:5]:
        if inst.feasible and inst.reference_solutions:
            mini.append(inst)
print(f'Mini set: {len(mini)} instructions')


In [ ]:
# Build mock outputs (echo first reference)
mock_outputs = []
for inst in mini:
    mock_outputs.append({
        'instruction_id': inst.id,
        'level': inst.level,
        'graph_serialized': inst.reference_solutions[0],
    })
print(f'Generated {len(mock_outputs)} mock outputs')


## 4. Score with D1 (structural) and D4 (instruction match)

These are the two cheapest metrics — no GPU, no BERT, no API.


In [ ]:
from graphinstruct.parser import parse
from graphinstruct.metrics.structural import valid_rate
from graphinstruct.metrics.instruction import instruction_score

import statistics

d1_per_inst = []
d4_per_inst = []
for inst, out in zip(mini, mock_outputs):
    try:
        result = parse(out['graph_serialized'])
        g = result.graph
        d1 = valid_rate([g], constraints=list(inst.explicit_constraints))
        d4 = instruction_score(g, list(inst.explicit_constraints))
    except Exception:
        d1, d4 = 0.0, 0.0
    d1_per_inst.append(d1)
    d4_per_inst.append(d4)

print(f'D1 (Valid Rate)         mean: {statistics.mean(d1_per_inst):.3f}')
print(f'D4 (Instruction Match)  mean: {statistics.mean(d4_per_inst):.3f}')
print('\nSince outputs == references, both should score near 1.0.')


## 5. What's next?

- **Try a real model**: see [`02_eval_your_model.ipynb`](02_eval_your_model.ipynb)
  for the recipe to plug in your own LLM
- **Reproduce paper numbers**: see [`docs/REPRODUCE.md`](../docs/REPRODUCE.md)
- **Inspect cached results**: open `results/quality/<model>-<strategy>.quality.json`
  for any of the 45 (model, strategy) cells in the paper
- **Run the D5 robustness ablation** (Appendix C, Tab. 3):
  ```bash
  python scripts/d5_robustness.py
  ```
